# Track B — H7 재검증: 튜닝된 최종 모델 기준 (track_b_09)

담당: 김태헌 | 선행 노트북: track_b_07(모델 튜닝·선정), track_b_08(H4 재검증)

## 이 노트북의 목적

track_b_05의 H7 결과("기존 신용정보 26개 중 25개가 SHAP 순위 유지 → 대안정보는 대체가 아닌 보완")는
**튜닝 전 고정 하이퍼파라미터 모델** 기준이었다. track_b_08에서 Model A(26개)·Model B(124개)를
각각 새로 튜닝했으므로, 이 노트북은 **그 튜닝된 두 모델의 SHAP 중요도를 직접 비교**해서 H7을
정식 재확정한다.

핵심 질문: *"대안변수를 추가해도, 기존 신용정보 26개 변수의 상대적 중요도 순위는 유지되는가?"*
- 유지된다 → 보완 관계 재확정
- 무너진다 → 대안정보가 기존정보를 밀어내는 대체 관계로 해석 변경 필요 (H7 기각)

## 입력

track_b_08에서 저장한 `track_b_h4_best_params.json`(Model A/B 최적 하이퍼파라미터)을 그대로
재사용해서 두 모델을 다시 학습한다(같은 데이터·같은 random_state이므로 재현 가능).


## 0. 환경 설정

In [4]:
# 로컬 환경: 터미널에서 아래 명령어로 한 번만 설치해두면 됨
# pip install "xgboost<3.0" shap

import warnings
warnings.filterwarnings('ignore')

import json
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
import xgboost as xgb
import shap

RANDOM_STATE = 42


## 1. 데이터 로드 및 병합 (track_b_08과 동일한 구성)

track_b_08과 완전히 동일한 방식으로 Model A / Model B 데이터를 재구성한다.
분할까지 동일한 `random_state`를 쓰므로, track_b_08에서 학습했던 것과 같은
train/test 분할이 재현된다.

In [5]:
handoff_path = r'C:\Users\tehun\Desktop\multicamp\project\creditscore\cardCB'  # 필요시 수정

df_h4 = pd.read_csv(f'{handoff_path}/track_b_traditional_credit_features_for_H4comparison.csv')
df_train = pd.read_csv(f'{handoff_path}/track_b_features_train_v2.csv')

assert set(df_h4['CUST_ID']) == set(df_train['CUST_ID']), "CUST_ID 불일치!"
assert (df_h4.set_index('CUST_ID')['TARGET'] == df_train.set_index('CUST_ID')['TARGET']).all(), "TARGET 불일치!"

traditional_cols = [c for c in df_h4.columns if c not in ['CUST_ID', 'TARGET']]
alt_cols = [c for c in df_train.columns if c not in ['CUST_ID', 'TARGET']]
categorical_cols = ['JB_TP', 'HOME_ADM']

df_merged = df_h4.merge(df_train.drop(columns=['TARGET']), on='CUST_ID', how='inner')
y = df_merged['TARGET']

X_A = df_merged[traditional_cols]
X_B_raw = df_merged[traditional_cols + alt_cols]
X_B = pd.get_dummies(X_B_raw, columns=categorical_cols, prefix=categorical_cols)

print(f"Model A: {X_A.shape}, Model B: {X_B.shape}")


Model A: (285890, 34), Model B: (285890, 124)


In [6]:
idx_train, idx_test = train_test_split(
    df_merged.index, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

X_A_train, X_A_test = X_A.loc[idx_train], X_A.loc[idx_test]
X_B_train, X_B_test = X_B.loc[idx_train], X_B.loc[idx_test]
y_train, y_test = y.loc[idx_train], y.loc[idx_test]

print(f"Train: {len(idx_train)} / Test: {len(idx_test)}")


Train: 228712 / Test: 57178


## 2. track_b_08 최적 하이퍼파라미터로 재학습

`track_b_h4_best_params.json`에 저장된 파라미터를 그대로 불러와서 두 모델을 다시 학습한다.
(RandomizedSearchCV를 다시 돌리지 않아도 되므로 훨씬 빠름)

In [7]:
with open(f'{handoff_path}/track_b_h4_best_params.json', encoding='utf-8') as f:
    h4_params = json.load(f)

params_A = h4_params['model_A_26vars']
params_B = h4_params['model_B_26plus69vars']

model_A = xgb.XGBClassifier(
    **params_A,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    eval_metric='aucpr', random_state=RANDOM_STATE, n_jobs=-1, tree_method='hist',
)
model_A.fit(X_A_train, y_train)

model_B = xgb.XGBClassifier(
    **params_B,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    eval_metric='aucpr', random_state=RANDOM_STATE, n_jobs=-1, tree_method='hist',
)
model_B.fit(X_B_train, y_train)

print("Model A / Model B 재학습 완료")


Model A / Model B 재학습 완료


## 3. SHAP 중요도 계산 (Model A, Model B 각각)

In [8]:
explainer_A = shap.TreeExplainer(model_A)
explainer_B = shap.TreeExplainer(model_B)

sample_n = 5000
X_A_sample = X_A_test.sample(min(sample_n, len(X_A_test)), random_state=RANDOM_STATE)
X_B_sample = X_B_test.loc[X_A_sample.index]  # 같은 고객으로 맞춰서 비교

shap_A = explainer_A.shap_values(X_A_sample)
shap_B = explainer_B.shap_values(X_B_sample)

# 평균 |SHAP| 기준 변수 중요도
imp_A = pd.Series(np.abs(shap_A).mean(axis=0), index=X_A_sample.columns).sort_values(ascending=False)
imp_B_full = pd.Series(np.abs(shap_B).mean(axis=0), index=X_B_sample.columns).sort_values(ascending=False)

print("Model A 상위 10개:")
print(imp_A.head(10))
print("\nModel B 상위 10개 (전체 124개 중):")
print(imp_B_full.head(10))


Model A 상위 10개:
PYE_C1L120012                      1.839791
PYE_C1L120196                      1.185773
CRDT_LN_BAL                        0.443566
PYE_C1L120161                      0.311788
PYE_L10231000                      0.229003
YOY_CRDT_LN_BAL_RTC_was_missing    0.222686
PYE_L102100CP                      0.201912
PYE_C1L120012_was_missing          0.192482
PYE_L1021003P                      0.157631
PYE_L10210800                      0.132005
dtype: float32

Model B 상위 10개 (전체 124개 중):
PYE_C1L120012                      1.403564
PYE_C1L120196                      0.699739
CRDT_LN_BAL                        0.421364
R6M_SSM_AMT                        0.300189
R6M_MED_AMT                        0.282133
PYE_L102100CP                      0.245647
YOY_CRDT_LN_BAL_RTC_was_missing    0.241325
PYE_L10231000                      0.197813
PYE_C1L120012_was_missing          0.190467
PYE_C1L120161                      0.186197
dtype: float32


## 4. H7 핵심 검증: 26개 변수의 순위가 유지되는가

Model B(124개) 안에서도 기존 신용정보 26개(+결측플래그)만 뽑아서, Model A 단독일 때의 순위와
비교한다. Track B 원 문서의 "34개 중 33개가 5계단 이내 유지" 기준을 그대로 적용.

In [9]:
rank_A = imp_A.rank(ascending=False)
rank_B_within_traditional = imp_B_full.loc[traditional_cols].rank(ascending=False)

rank_compare = pd.DataFrame({
    'rank_in_ModelA': rank_A,
    'rank_in_ModelB(26개만 추출)': rank_B_within_traditional,
}).dropna()
rank_compare['순위_변화'] = rank_compare['rank_in_ModelB(26개만 추출)'] - rank_compare['rank_in_ModelA']
rank_compare['5계단_이내_유지'] = rank_compare['순위_변화'].abs() <= 5
rank_compare = rank_compare.sort_values('rank_in_ModelA')

n_total = len(rank_compare)
n_stable = rank_compare['5계단_이내_유지'].sum()
stable_pct = n_stable / n_total * 100

print(f"H7 재검증 결과: {n_total}개 중 {n_stable}개가 5계단 이내 순위 유지 ({stable_pct:.1f}%)")
print(f"track_b_05 원 결과(고정 파라미터 기준): 34개 중 33개 유지 (97.1%)")
print()
rank_compare


H7 재검증 결과: 34개 중 33개가 5계단 이내 순위 유지 (97.1%)
track_b_05 원 결과(고정 파라미터 기준): 34개 중 33개 유지 (97.1%)



,rank_in_ModelA,rank_in_ModelB(26개만 추출),순위_변화,5계단_이내_유지
PYE_C1L120012,1.0,1.0,0.0,True
PYE_C1L120196,2.0,2.0,0.0,True
CRDT_LN_BAL,3.0,3.0,0.0,True
PYE_C1L120161,4.0,8.0,4.0,True
PYE_L10231000,5.0,6.0,1.0,True
YOY_CRDT_LN_BAL_RTC_was_missing,6.0,5.0,-1.0,True
PYE_L102100CP,7.0,4.0,-3.0,True
PYE_C1L120012_was_missing,8.0,7.0,-1.0,True
PYE_L1021003P,9.0,11.0,2.0,True
PYE_L10210800,10.0,10.0,0.0,True


## 5. H7 판정

- **유지율이 track_b_05와 비슷하거나 높으면** → H7 재확정 (보완 관계 결론 그대로 유지)
- **유지율이 크게 낮아졌으면** → 대안변수 추가로 XGBoost가 기존 신용정보 대신 대안변수를 더 강하게
  쓰는 방향으로 재배치됐다는 뜻 → "보완"이 아니라 "부분 대체"로 해석을 수정하고, 어떤 변수가
  순위 이탈했는지(아래 셀) 확인해서 원인 서술 필요

In [10]:
dropped = rank_compare[~rank_compare['5계단_이내_유지']].sort_values('순위_변화', ascending=False)
print("순위 변동이 5계단을 넘은 변수:")
dropped


순위 변동이 5계단을 넘은 변수:


,rank_in_ModelA,rank_in_ModelB(26개만 추출),순위_변화,5계단_이내_유지
PYE_L10216800,33.0,24.0,-9.0,False


## 6. 결과 저장

In [11]:
rank_compare.to_csv(f'{handoff_path}/track_b_h7_revalidation_final.csv', encoding='utf-8-sig')

h7_summary = {
    'n_total_traditional_vars': int(n_total),
    'n_rank_stable_within_5': int(n_stable),
    'stable_pct': float(stable_pct),
    'track_b_05_reference_pct': 97.1,  # 34개 중 33개, 참고용
    'verdict': 'H7 재확정(보완 관계 유지)' if stable_pct >= 90 else '재검토 필요(부분 대체 가능성)',
}

with open(f'{handoff_path}/track_b_h7_summary.json', 'w', encoding='utf-8') as f:
    json.dump(h7_summary, f, ensure_ascii=False, indent=2)

print("저장 완료: track_b_h7_revalidation_final.csv, track_b_h7_summary.json")
print(h7_summary)


저장 완료: track_b_h7_revalidation_final.csv, track_b_h7_summary.json
{'n_total_traditional_vars': 34, 'n_rank_stable_within_5': 33, 'stable_pct': 97.05882352941177, 'track_b_05_reference_pct': 97.1, 'verdict': 'H7 재확정(보완 관계 유지)'}
